# 03 — Neural Baselines (CNN / BiLSTM+Attention) — Multi-Task Ticket Sınıflandırma

Bu notebook, transformer (XLM-RoBERTa) öncesi sıfırdan eğitilen embedding tabanlı
multi-task baseline modellerini içerir: **CNN** ve **BiLSTM + Attention**.

Bu revizyonda yapılan iyileştirmeler:
- Tekrarlanabilirlik için tam seed sabitleme (numpy/torch/random + DataLoader generator)
- Windows'a özel sabit path'ler kaldırıldı, ortam-bağımsız hale getirildi
- `subject` alanı da girdi metnine dahil edildi (2.673 kayıtta boş, `fillna("")` ile güvenli)
- Türkçe'ye duyarlı casefold (İ/I/ı ayrımı) eklendi — `str.lower()` Türkçe'de "İstanbul" -> "i̇stanbul" gibi hatalı sonuçlar üretir
- İşlenmiş split dosyaları yoksa ham veriden stratified split üreten güvenli fallback
- Sadece accuracy yerine **macro-F1** ve tam `classification_report` metrikleri
- **priority** görevi için dil bazında (en/de/tr) ayrı performans raporu — çünkü EDA'da
  `critical` etiketinin SADECE Türkçe örneklerde bulunduğu doğrulandı (2.100/2.100).
  Bu, modelin "kritik" tahminini dilden (örn. Türkçe kelime dağarcığından) öğrenebileceği
  anlamına gelir; bu nedenle croslingual genelleme metrikleri ayrı raporlanmalı.
- Gradient clipping + ReduceLROnPlateau learning-rate scheduler
- En iyi model seçimi artık val_loss yerine **görevler arası ortalama macro-F1**'e göre
- BiLSTM'e additive attention pooling eklendi (son gizli durum yerine tüm zaman adımlarına dikkat)
- Embedding dropout eklendi (overfitting azaltmak için)
- Model/mlflow dizinleri otomatik oluşturuluyor (`FileNotFoundError` riski kaldırıldı)

## 1. Kurulum, Reproducibility ve Cihaz

In [1]:
"""torch import edilmeden ONCE cagrilmasi gerekir."""
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import os
import re
import json
import random
import sys
import sysconfig
import platform
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

def _prepare_windows_dll_path():
    """Bazi Windows + venv kurulumlarinda torch'un DLL bagimliliklari (c10.dll vb.)
    standart arama yolunda bulunamiyor (WinError 1114). Orijinal notebook'ta bu,
    kullaniciya ("Mustafa") ve venv'e ozel sabit bir path ile cozulmustu. Burada
    ayni cozum, aktif Python yorumlayicisinin kendi site-packages ve Scripts
    klasorlerinden otomatik turetiliyor - baska bir makinede/venv'de de calisir."""
    if platform.system() != "Windows":
        return
    site_packages = Path(sysconfig.get_paths()["purelib"])
    torch_lib_path = site_packages / "torch" / "lib"
    scripts_path = Path(sys.prefix) / "Scripts"
    os.environ["PATH"] = str(torch_lib_path) + os.pathsep + str(scripts_path) + os.pathsep + os.environ.get("PATH", "")
    if torch_lib_path.exists():
        os.add_dll_directory(str(torch_lib_path))
    if scripts_path.exists():
        os.add_dll_directory(str(scripts_path))

_prepare_windows_dll_path()

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report
import mlflow

SEED = 42

def seed_everything(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

TASK_COLS = ["type", "queue", "category", "priority"]

MLFLOW_DB_PATH = Path("../mlflow.db")
MLFLOW_DB_PATH.parent.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB_PATH.as_posix()}")
mlflow.set_experiment("neural_baselines")

MODEL_DIR = Path("../src/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

c:\Users\Mustafa\s_env\Lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
c:\Users\Mustafa\s_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## 2. Veri Yükleme

Önce işlenmiş `train.jsonl` / `val.jsonl` split dosyaları aranır (önceki notebook'un
ürettiği stratified split). Bulunamazsa, ham birleşik veri setinden (`RAW_PATH`) aynı
stratejiyle (priority'ye göre stratified, %90/%10) bir split otomatik üretilir — böylece
notebook, veri hazırlama adımı çalıştırılmamış bir ortamda da güvenle çalışır.

In [2]:
PROCESSED_TRAIN = Path("../data/processed/train.jsonl")
PROCESSED_VAL = Path("../data/processed/val.jsonl")
RAW_PATH = Path("../data/raw/train.jsonl")  # tek parça ham veri (fallback kaynağı)

# Türkçe'ye duyarlı casefold: str.lower() Türkçe İ/I harflerinde hatalı sonuç verir.
_TR_UPPER_MAP = str.maketrans({"İ": "i", "I": "ı", "Ş": "ş", "Ğ": "ğ", "Ü": "ü", "Ö": "ö", "Ç": "ç"})

def turkish_casefold(t: str) -> str:
    return t.translate(_TR_UPPER_MAP).casefold()

def light_clean(t):
    if not isinstance(t, str):
        return ""
    t = t.replace("\\n", " ").replace("\n", " ")
    t = re.sub(r"<[^>]+>", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def build_text_column(df: pd.DataFrame) -> pd.DataFrame:
    # subject 2.673 kayıtta boş -> fillna ile güvenli birlestirme.
    # subject + body birlikte kullanmak type/queue/category için ek sinyal saglar.
    df = df.copy()
    df["subject"] = df.get("subject", pd.Series([""] * len(df))).fillna("")
    df["text"] = (df["subject"] + " . " + df["body"]).apply(light_clean)
    return df

if PROCESSED_TRAIN.exists() and PROCESSED_VAL.exists():
    train_df = pd.read_json(PROCESSED_TRAIN, lines=True)
    val_df = pd.read_json(PROCESSED_VAL, lines=True)
    print("Islenmis split dosyalari bulundu, dogrudan yuklendi.")
else:
    print("Islenmis split bulunamadi, ham veriden stratified split uretiliyor:", RAW_PATH)
    full_df = pd.read_json(RAW_PATH, lines=True)
    train_df, val_df = train_test_split(
        full_df, test_size=0.1, random_state=SEED, stratify=full_df["priority"]
    )
    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

train_df = build_text_column(train_df)
val_df = build_text_column(val_df)
train_df["body_light_clean"] = train_df["text"]
val_df["body_light_clean"] = val_df["text"]

print("train:", train_df.shape, " val:", val_df.shape)

Islenmis split dosyalari bulundu, dogrudan yuklendi.
train: (22927, 20)  val: (4912, 20)


## 3. Hızlı Veri Sağlık Kontrolü (EDA özeti)

Fine-tune öncesi, veri setinin bilinen kritik sorununu doğrulamak için hızlı bir kontrol:
`priority=critical` etiketinin dil dağılımı. Bu, modelin per-language performansının
ayrı raporlanması gerektiğinin gerekçesidir (aşağıda eğitim döngüsünde uygulanır).

In [3]:
print("=== priority x language dagilimi (train) ===")
print(pd.crosstab(train_df["priority"], train_df["language"]))

print("\n=== Gorev bazinda sinif dagilimi (train, oran) ===")
for col in TASK_COLS:
    print(f"--- {col} ---")
    print((train_df[col].value_counts(normalize=True) * 100).round(1))

=== priority x language dagilimi (train) ===
language    de    en    tr
priority                  
critical     0     0  2100
high      3362  4452   443
low       1768  2338   188
medium    3443  4603   230

=== Gorev bazinda sinif dagilimi (train, oran) ===
--- type ---
type
Incident    38.2
Request     31.5
Problem     20.6
Change       9.8
Name: proportion, dtype: float64
--- queue ---
queue
Technical Support                  26.4
Product Support                    16.9
Customer Service                   13.9
IT Support                         11.4
Billing and Payments                9.4
Returns and Exchanges               5.3
Service Outages and Maintenance     5.2
Sales and Pre-Sales                 4.1
Human Resources                     3.7
General Inquiry                     3.7
Name: proportion, dtype: float64
--- category ---
category
Technical Issue       59.9
Account Management    17.6
Billing                9.4
General Inquiry        7.8
Refund                 5.3
Name: pr

## 4. Etiket Kodlama ve Sınıf Ağırlıkları

In [4]:
encoders = {}
y_train_dict, y_val_dict, class_weights = {}, {}, {}

for col in TASK_COLS:
    le = LabelEncoder()
    y_train_dict[col] = le.fit_transform(train_df[col])
    y_val_dict[col] = le.transform(val_df[col])
    encoders[col] = le

    classes = np.arange(len(le.classes_))
    w = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_dict[col])
    class_weights[col] = torch.tensor(w, dtype=torch.float32).to(device)
    print(col, "-> num_classes:", len(le.classes_), " classes:", list(le.classes_))

type -> num_classes: 4  classes: ['Change', 'Incident', 'Problem', 'Request']
queue -> num_classes: 10  classes: ['Billing and Payments', 'Customer Service', 'General Inquiry', 'Human Resources', 'IT Support', 'Product Support', 'Returns and Exchanges', 'Sales and Pre-Sales', 'Service Outages and Maintenance', 'Technical Support']
category -> num_classes: 5  classes: ['Account Management', 'Billing', 'General Inquiry', 'Refund', 'Technical Issue']
priority -> num_classes: 4  classes: ['critical', 'high', 'low', 'medium']


## 5. Tokenizasyon ve Vocab

In [5]:
VOCAB_SIZE = 20000
SEQ_LEN = 128  # ortalama body ~380 karakter (~60-70 kelime); subject+body icin 128 daha guvenli

def simple_tokenize(text):
    return re.findall(r"\w+", turkish_casefold(text))

counter = Counter()
for t in train_df["body_light_clean"]:
    counter.update(simple_tokenize(t))

vocab = {"<pad>": 0, "<unk>": 1}
# most_common esit frekanslarda kararsiz siralama verebilir; deterministik olmasi icin
# (frekans azalan, kelime alfabetik artan) seklinde sirala.
for word, _ in sorted(counter.items(), key=lambda kv: (-kv[1], kv[0]))[: VOCAB_SIZE - 2]:
    vocab[word] = len(vocab)

print("Vocab boyutu:", len(vocab))

def encode(text, vocab, seq_len):
    tokens = simple_tokenize(text)
    ids = [vocab.get(tok, vocab["<unk>"]) for tok in tokens[:seq_len]]
    return ids + [vocab["<pad>"]] * (seq_len - len(ids))

Vocab boyutu: 20000


## 6. Dataset / DataLoader

In [6]:
class TicketDataset(Dataset):
    def __init__(self, texts, y_dict, vocab, seq_len):
        self.texts = texts
        self.y_dict = y_dict
        self.vocab = vocab
        self.seq_len = seq_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = encode(self.texts[idx], self.vocab, self.seq_len)
        item = {"input_ids": torch.tensor(ids, dtype=torch.long)}
        for col in TASK_COLS:
            item[col] = torch.tensor(self.y_dict[col][idx], dtype=torch.long)
        return item

train_ds = TicketDataset(train_df["body_light_clean"].values, y_train_dict, vocab, SEQ_LEN)
val_ds = TicketDataset(val_df["body_light_clean"].values, y_val_dict, vocab, SEQ_LEN)

# Shuffle'in da deterministik olmasi icin ayri bir generator kullaniliyor.
_g = torch.Generator()
_g.manual_seed(SEED)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, generator=_g)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

# priority icin dil-bazli degerlendirme yapabilmek amaciyla val siralamasini sabit tutuyoruz
val_languages = val_df["language"].values

## 7. Model Mimarileri

`ModuleDict` anahtarlarında görev isimlerini (örn. `"type"`) doğrudan kullanmıyoruz —
`nn.Module`'ün yerleşik `.type()` metoduyla çakışıyor. Bu yüzden `head_` prefix'i zorunlu.

**CNN**: TextCNN, çoklu çekirdek boyutlu 1D konvolüsyon + max-pool.
**BiLSTM+Attention**: Önceki sürümde sadece son gizli durum kullanılıyordu; bu, uzun
metinlerde bilgi kaybına yol açar. Şimdi tüm zaman adımlarına additive attention
uygulanıyor, böylece model önemli kelimelere (örn. "kritik", "acil", "iade") daha
doğrudan ağırlık verebiliyor.

In [7]:
class MultiTaskCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes_dict, kernel_sizes=(3, 4, 5),
                 num_filters=64, emb_dropout=0.1):
        super().__init__()
        self.task_names = list(num_classes_dict.keys())
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.emb_dropout = nn.Dropout(emb_dropout)
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, num_filters, kernel_size=k) for k in kernel_sizes])
        self.dropout = nn.Dropout(0.3)
        self.shared = nn.Linear(num_filters * len(kernel_sizes), 128)
        self.heads = nn.ModuleDict({
            f"head_{task}": nn.Linear(128, n) for task, n in num_classes_dict.items()
        })

    def forward(self, input_ids):
        x = self.emb_dropout(self.embedding(input_ids)).permute(0, 2, 1)
        conv_outs = [torch.max(torch.relu(conv(x)), dim=2).values for conv in self.convs]
        x = self.dropout(torch.cat(conv_outs, dim=1))
        shared = self.dropout(torch.relu(self.shared(x)))
        return {task: self.heads[f"head_{task}"](shared) for task in self.task_names}


class AdditiveAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.proj = nn.Linear(hidden_dim, hidden_dim)
        self.score = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, sequence, mask):
        # sequence: (B, T, H), mask: (B, T) 1=gecerli token, 0=pad
        energy = torch.tanh(self.proj(sequence))
        scores = self.score(energy).squeeze(-1)          # (B, T)
        scores = scores.masked_fill(mask == 0, float("-inf"))
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)  # (B, T, 1)
        context = (sequence * weights).sum(dim=1)          # (B, H)
        return context


class MultiTaskBiLSTMAttn(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes_dict, emb_dropout=0.1):
        super().__init__()
        self.task_names = list(num_classes_dict.keys())
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.emb_dropout = nn.Dropout(emb_dropout)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.attn = AdditiveAttention(hidden_dim * 2)
        self.dropout = nn.Dropout(0.3)
        self.shared = nn.Linear(hidden_dim * 2, 128)
        self.heads = nn.ModuleDict({
            f"head_{task}": nn.Linear(128, n) for task, n in num_classes_dict.items()
        })

    def forward(self, input_ids):
        mask = (input_ids != 0).long()
        x = self.emb_dropout(self.embedding(input_ids))
        seq_out, _ = self.lstm(x)                # (B, T, 2*hidden)
        context = self.attn(seq_out, mask)
        context = self.dropout(context)
        shared = self.dropout(torch.relu(self.shared(context)))
        return {task: self.heads[f"head_{task}"](shared) for task in self.task_names}

## 8. Eğitim Döngüsü

Değişenler:
- `clip_grad_norm_` ile gradient clipping (patlayan gradyanlara karşı)
- `ReduceLROnPlateau` scheduler (val_loss plato yaptığında lr düşürülür)
- Her epoch sonunda macro-F1 hesaplanır ve en iyi model **ortalama macro-F1**'e göre seçilir
  (accuracy tek başına, `medium`/`high` gibi baskın sınıflara karşı yanıltıcıdır)
- Eğitim bitince `priority` için ayrıca **dil bazında** classification_report basılır —
  `critical` etiketinin yalnızca Türkçe örneklerde bulunması nedeniyle bu görevin
  cross-lingual performansı ayrı değerlendirilmelidir

In [8]:
num_classes_dict = {col: len(encoders[col].classes_) for col in TASK_COLS}

def evaluate(model, loader):
    model.eval()
    val_loss = 0.0
    all_preds = {col: [] for col in TASK_COLS}
    all_true = {col: [] for col in TASK_COLS}
    criterion = {col: nn.CrossEntropyLoss(weight=class_weights[col]) for col in TASK_COLS}
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            outputs = model(input_ids)
            loss = sum(criterion[col](outputs[col], batch[col].to(device)) for col in TASK_COLS)
            val_loss += loss.item()
            for col in TASK_COLS:
                preds = outputs[col].argmax(dim=1).cpu().numpy()
                all_preds[col].extend(preds)
                all_true[col].extend(batch[col].numpy())
    val_loss /= len(loader)
    macro_f1 = {col: f1_score(all_true[col], all_preds[col], average="macro") for col in TASK_COLS}
    return val_loss, macro_f1, all_preds, all_true


def train_model(model, name, epochs=15, patience=3, lr=1e-3):
    model.to(device)
    criterion = {col: nn.CrossEntropyLoss(weight=class_weights[col]) for col in TASK_COLS}
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1)

    best_avg_f1 = -1.0
    patience_counter = 0
    best_state = None

    with mlflow.start_run(run_name=name):
        mlflow.log_param("architecture", name)
        mlflow.log_param("vocab_size", VOCAB_SIZE)
        mlflow.log_param("seq_len", SEQ_LEN)
        mlflow.log_param("lr", lr)
        mlflow.log_param("seed", SEED)

        for epoch in range(epochs):
            model.train()
            train_loss = 0.0
            for batch in train_loader:
                input_ids = batch["input_ids"].to(device)
                optimizer.zero_grad()
                outputs = model(input_ids)
                loss = sum(criterion[col](outputs[col], batch[col].to(device)) for col in TASK_COLS)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()
                train_loss += loss.item()
            train_loss /= len(train_loader)

            val_loss, macro_f1, _, _ = evaluate(model, val_loader)
            avg_f1 = sum(macro_f1.values()) / len(macro_f1)
            scheduler.step(val_loss)

            print(f"[{name}] epoch {epoch+1}/{epochs} train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
                  + " ".join(f"{col}_macroF1={macro_f1[col]:.3f}" for col in TASK_COLS)
                  + f" avg_macroF1={avg_f1:.3f}")

            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("val_loss", val_loss, step=epoch)
            mlflow.log_metric("avg_macro_f1", avg_f1, step=epoch)
            for col in TASK_COLS:
                mlflow.log_metric(f"val_{col}_macro_f1", macro_f1[col], step=epoch)

            if avg_f1 > best_avg_f1:
                best_avg_f1 = avg_f1
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping (epoch {epoch+1})")
                    break

        model.load_state_dict(best_state)
        torch.save(model.state_dict(), MODEL_DIR / f"{name}.pt")

        # Final rapor: gorev basina classification_report + priority icin dil-bazli kirilim
        _, final_f1, final_preds, final_true = evaluate(model, val_loader)
        for col in TASK_COLS:
            report = classification_report(
                final_true[col], final_preds[col],
                target_names=encoders[col].classes_, zero_division=0,
            )
            print(f"\n--- {name} | {col} classification report ---\n{report}")
            mlflow.log_text(report, f"{col}_classification_report.txt")

        print(f"\n--- {name} | priority: dil bazinda kirilim ---")
        priority_preds = np.array(final_preds["priority"])
        priority_true = np.array(final_true["priority"])
        for lang in sorted(set(val_languages)):
            lang_mask = val_languages == lang
            if lang_mask.sum() == 0:
                continue
            lang_f1 = f1_score(priority_true[lang_mask], priority_preds[lang_mask], average="macro", zero_division=0)
            print(f"  {lang}: n={lang_mask.sum():5d}  macro_f1={lang_f1:.3f}")
            mlflow.log_metric(f"priority_macro_f1_{lang}", lang_f1)

    return model

## 9. Eğitim: CNN ve BiLSTM+Attention

In [9]:
cnn_model = MultiTaskCNN(len(vocab), embed_dim=128, num_classes_dict=num_classes_dict)
cnn_model = train_model(cnn_model, "cnn_multitask")

rnn_model = MultiTaskBiLSTMAttn(len(vocab), embed_dim=128, hidden_dim=64, num_classes_dict=num_classes_dict)
rnn_model = train_model(rnn_model, "bilstm_attn_multitask")

[cnn_multitask] epoch 1/15 train_loss=5.6669 val_loss=5.0287 type_macroF1=0.628 queue_macroF1=0.219 category_macroF1=0.417 priority_macroF1=0.431 avg_macroF1=0.424
[cnn_multitask] epoch 2/15 train_loss=4.8998 val_loss=4.7698 type_macroF1=0.685 queue_macroF1=0.269 category_macroF1=0.415 priority_macroF1=0.443 avg_macroF1=0.453
[cnn_multitask] epoch 3/15 train_loss=4.6612 val_loss=4.5888 type_macroF1=0.722 queue_macroF1=0.341 category_macroF1=0.460 priority_macroF1=0.464 avg_macroF1=0.497
[cnn_multitask] epoch 4/15 train_loss=4.4495 val_loss=4.4793 type_macroF1=0.655 queue_macroF1=0.351 category_macroF1=0.484 priority_macroF1=0.480 avg_macroF1=0.493
[cnn_multitask] epoch 5/15 train_loss=4.2941 val_loss=4.3905 type_macroF1=0.727 queue_macroF1=0.396 category_macroF1=0.508 priority_macroF1=0.483 avg_macroF1=0.529
[cnn_multitask] epoch 6/15 train_loss=4.1757 val_loss=4.3284 type_macroF1=0.730 queue_macroF1=0.437 category_macroF1=0.525 priority_macroF1=0.504 avg_macroF1=0.549
[cnn_multitask] 

## 10. Model Karşılaştırması

Her iki modelin görev başına macro-F1 skorlarını yan yana özetler.

In [10]:
comparison_rows = []
for name, model in [("cnn_multitask", cnn_model), ("bilstm_attn_multitask", rnn_model)]:
    _, macro_f1, _, _ = evaluate(model, val_loader)
    row = {"model": name, **{f"{col}_macro_f1": round(macro_f1[col], 4) for col in TASK_COLS}}
    row["avg_macro_f1"] = round(sum(macro_f1.values()) / len(macro_f1), 4)
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows).set_index("model")
comparison_df

,type_macro_f1,queue_macro_f1,category_macro_f1,priority_macro_f1,avg_macro_f1
model,,,,,
cnn_multitask,0.7381,0.4821,0.5952,0.5127,0.5820
bilstm_attn_multitask,0.7866,0.4972,0.6197,0.5389,0.6106


---
**Not (veri kalitesi uyarısı):** `priority=critical` etiketi veri setinde yalnızca Türkçe
kayıtlarda bulunuyor (2.100/2.100). Bu, `priority` göreviyle ilgili genel macro-F1'in
büyük ölçüde "Türkçe metni tanıma"dan kaynaklanabileceği anlamına gelir; modelin gerçekten
aciliyet sinyalini mi yoksa dil sinyalini mi öğrendiğini ayırt etmek için yukarıdaki
dil-bazlı kırılım raporuna bakın. Kalıcı çözüm için üretim/etiketleme sürecinde
`critical` etiketinin diğer dillerde de örneklenmesi (veya en azından değerlendirmenin
her zaman dil kontrollü yapılması) önerilir.